# Historical reproduction

This notebook reproduces the archived 2025 evaluation in a fresh work directory. It does not fit the versioned measurement study. Set `BDB_EXECUTE=1` to run extraction through reporting. Display mode only reads the archive.

Kaggle execution status: not run. No Kaggle mount was used.


In [ ]:
from pathlib import Path
import os, json, sys
import pandas as pd
import numpy as np

ROOT = Path.cwd().resolve()
if not (ROOT / "config" / "research.yaml").exists():
    for parent in [ROOT, *ROOT.parents]:
        if (parent / "config" / "research.yaml").exists():
            os.chdir(parent)
            ROOT = parent
            break
os.environ["BDB2027_ROOT"] = str(ROOT)
print("project", ROOT)

candidates = [
    Path(os.environ["BDB_DATA_ROOT"]) if os.environ.get("BDB_DATA_ROOT") else None,
    ROOT / "nfl-big-data-bowl-2027",
    Path("/kaggle/input/nfl-big-data-bowl-2027"),
]
data = next(p for p in candidates if p is not None and (p / "players.csv").exists())
os.environ["BDB_DATA_ROOT"] = str(data)
print("data", data)
print("csv count", len(list(data.glob("*.csv"))))


In [ ]:
src = str(ROOT / "src")
os.environ["PYTHONPATH"] = src + os.pathsep + os.environ.get("PYTHONPATH", "")
if src not in sys.path:
    sys.path.insert(0, src)
import bdb2027
from bdb2027.config import Project
probe = Project(data_root=data)
assert probe.data_root == data
print("import", Path(bdb2027.__file__).resolve())
print("child path will include", src)
manifest = json.loads((ROOT / "archive/original_evaluation/freeze_manifest.json").read_text())
print("historical frozen_at", manifest["frozen_at"])
print("historical code_hash", manifest["code_hash"])
print("historical players", manifest["n_train_players"], "routes", manifest["n_train_routes"])


In [ ]:
import subprocess
executed = os.environ.get("BDB_EXECUTE") == "1"
work = Path(os.environ.get("BDB2027_WORK", f"/tmp/bdb_hist_repro_{os.getpid()}"))
if executed:
    if work.exists() and any(work.iterdir()):
        raise RuntimeError(f"{work} is not an empty work directory")
    work.mkdir(parents=True, exist_ok=True)
    env = os.environ.copy()
    env["PYTHONPATH"] = str(ROOT / "src") + os.pathsep + env.get("PYTHONPATH", "")
    env["BDB2027_WORK"] = str(work)
    env["BDB2027_ROOT"] = str(ROOT)
    env["BDB_DATA_ROOT"] = str(data)
    py = sys.executable
    steps = [
        [py, "-m", "bdb2027", "--data-root", str(data), "extract", "--cohorts", "2023", "2024"],
        [py, "-m", "bdb2027", "--data-root", str(data), "protocol"],
        [py, "-m", "bdb2027", "--data-root", str(data), "develop"],
        [py, "-m", "bdb2027", "--data-root", str(data), "freeze"],
        [py, "-m", "bdb2027", "--data-root", str(data), "extract", "--cohorts", "2025"],
        [py, "-m", "bdb2027", "--data-root", str(data), "evaluate-heldout", "--cohort", "2025"],
        [py, "-m", "bdb2027", "--data-root", str(data), "report"],
    ]
    for step in steps:
        print("RUN", " ".join(step[3:]))
        subprocess.check_call(step, env=env, cwd=str(ROOT))
    pred_path = work / "outputs" / "predictions.csv"
else:
    pred_path = None
    print("Display mode. Historical file will be read. Full reproduction was not started in this process.")
print("prediction path", pred_path)


In [ ]:
archive = pd.read_csv(ROOT / "archive/original_evaluation/predictions.csv")
historical = archive.loc[archive["split"] == "holdout_2025"]
hist_players = historical.groupby("nfl_id").agg(y=("y", "mean"), m1=("pred_M1", "mean"), m2=("pred_M2", "mean"))
hist_delta = float((hist_players.m2 - hist_players.y).abs().mean() - (hist_players.m1 - hist_players.y).abs().mean())
print("historical 2025 delta", hist_delta)
print("historical players", historical["nfl_id"].nunique(), "routes", len(historical))
if pred_path is not None:
    fresh = pd.read_csv(pred_path)
    fresh_hold = fresh.loc[fresh["split"] == "holdout_2025"]
    merged = historical.merge(fresh_hold, on=["nfl_id", "game_id", "play_id"], suffixes=("_hist", "_repro"))
    tolerance = 1e-8
    for col in ["pred_M0", "pred_M1", "pred_M2", "y"]:
        diff = (merged[f"{col}_hist"] - merged[f"{col}_repro"]).abs().max()
        print(col, "max abs diff", diff)
        assert diff <= tolerance, (col, diff)
    repro_players = fresh_hold.groupby("nfl_id").agg(y=("y", "mean"), m1=("pred_M1", "mean"), m2=("pred_M2", "mean"))
    repro_delta = float((repro_players.m2 - repro_players.y).abs().mean() - (repro_players.m1 - repro_players.y).abs().mean())
    print("reproduced 2025 delta", repro_delta)
    print("These two lines are the archived result and the fresh-directory result, printed separately.")
    assert abs(hist_delta - repro_delta) <= tolerance
else:
    print("No fresh predictions in this process.")


The versioned SPEED_OUT/DAGGER study is not run in this notebook. Its configuration is `config/measurement_v2.yaml` and its outputs belong in a different work directory.
